# 1741. Find Total Time Spent by Each Employee

## Problem Description

An employee's daily office activity may contain several arrival and departure sessions.

### Table Schema

**Employees**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| emp_id | int | employee identifier |
| event_day | date | day of attendance |
| in_time | int | arrival time in minutes |
| out_time | int | departure time in minutes |

### Task

For each employee and day, calculate the total minutes spent in the office by summing out_time minus in_time over that day's sessions. Do not use the difference between the day's earliest arrival and latest departure, which would include gaps. Return day (event_day renamed), emp_id, total_time. Any row order is accepted.

## Example

### Input

**employees**

| emp_id | event_day | in_time | out_time |
| --- | --- | --- | --- |
| 1 | 2024-01-01 | 10 | 20 |
| 1 | 2024-01-01 | 25 | 40 |
| 1 | 2024-01-02 | 5 | 6 |
| 2 | 2024-01-01 | 5 | 15 |

### Expected Output

| day | emp_id | total_time |
| --- | --- | --- |
| 2024-01-01 | 1 | 25 |
| 2024-01-02 | 1 | 1 |
| 2024-01-01 | 2 | 10 |

### Explanation

Employee 1 spends 10 + 15 = 25 minutes on January 1 and 1 minute on January 2. Employee 2 spends 10 minutes on January 1.

## Practice

The next code cell constructs exactly these example DataFrames. Try the blank practice cell before reading the solutions.

## References

The problem is restated for self-contained practice; the examples are original. Community links are for further reading, and implementations below are independently written.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/1700-1799/1741.Find%20Total%20Time%20Spent%20by%20Each%20Employee/README_EN.md) · [LeetCode](https://leetcode.com/problems/find-total-time-spent-by-each-employee/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/find-total-time-spent-by-each-employee/solutions/)

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal

employees = pd.DataFrame([[1,"2024-01-01",10,20],[1,"2024-01-01",25,40],[1,"2024-01-02",5,6],[2,"2024-01-01",5,15]], columns=["emp_id","event_day","in_time","out_time"])
expected = pd.DataFrame([["2024-01-01",1,25],["2024-01-02",1,1],["2024-01-01",2,10]], columns=["day","emp_id","total_time"])
display(employees)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Derived column and groupby
先新增 duration，再依兩個鍵加總。

In [ ]:
def solution_1(employees):
    return employees.assign(total_time=employees.out_time-employees.in_time).groupby(['event_day','emp_id'],as_index=False).total_time.sum().rename(columns={'event_day':'day'})

result_1 = solution_1(employees.copy())
display(result_1)

### Solution 2 — Aggregate both totals
sum(out-in)=sum(out)-sum(in)，假設時間皆非空值。

In [ ]:
def solution_2(employees):
    g = employees.groupby(['event_day','emp_id'],as_index=False).agg(start=('in_time','sum'),end=('out_time','sum'))
    return g.assign(total_time=g.end-g.start).rename(columns={'event_day':'day'})[['day','emp_id','total_time']]

result_2 = solution_2(employees.copy())
display(result_2)

### Solution 3 — Pivot table
pivot_table 可進一步轉成矩陣報表。

In [ ]:
def solution_3(employees):
    p = employees.assign(total_time=employees.out_time-employees.in_time).pivot_table(index=['event_day','emp_id'],values='total_time',aggfunc='sum')
    return p.reset_index().rename(columns={'event_day':'day'})

result_3 = solution_3(employees.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare row grain, missing matches, duplicate keys and zero-count categories.